In [36]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [37]:
SEQUENCE_LENGTH = 50
NUM_FEATURES = 18

In [ ]:
current_dir = Path.cwd()
gesture_data_dir = current_dir.parent.parent / "data" / "gesture"
gesture_model_dir = current_dir.parent.parent / "models" / "gesture"

In [39]:
def resample_sequence(df: pd.DataFrame, target_length: int) -> pd.DataFrame:
        """
        Adjusts the number of rows in the DataFrame to target_length.
        If there are fewer rows, interpolation is performed.
        If there are more rows, points are selected uniformly.

        Args:
            df: Pandas DataFrame containing time-series data (one gesture).
            target_length: Desired number of points after resampling.

        Returns:
            Pandas DataFrame with the same structure and target_length rows.
        """
        df = df.reset_index(drop=True)
        current_length = len(df)

        if current_length < target_length:
            new_index = np.linspace(0, current_length - 1, target_length)
            df_resampled = df.reindex(new_index)
            df_resampled = df_resampled.interpolate(method="linear")
            return df_resampled.reset_index(drop=True)

        elif current_length > target_length:
            indices = np.linspace(0, current_length - 1, target_length, dtype=int)
            df_resampled = df.iloc[indices].reset_index(drop=True)
            return df_resampled

        else:
            return df

In [40]:
samples = []
labels = []

for file in os.listdir(gesture_data_dir):
    if file.endswith(".csv"):
        label = file.split("_")[0]
        
        df = pd.read_csv(os.path.join(gesture_data_dir, file))
        df_resampled = resample_sequence(df, SEQUENCE_LENGTH)
        
        if df_resampled.shape != (SEQUENCE_LENGTH, NUM_FEATURES):
            print(f"Skipping {file} after resampling — got {df_resampled.shape}")
            continue
        
        samples.append(df_resampled.values.astype(float))
        labels.append(label)

print(f"Loaded {len(samples)} gestures.")

Loaded 228 gestures.


In [ ]:
samples = np.array(samples)
labels = np.array(labels)


X_train, X_test, labels_train, labels_test = train_test_split(
    samples,
    labels,
    test_size=0.3,
    random_state=42,
    stratify=labels,
)


encoder = OneHotEncoder(sparse_output=False)

y_train = encoder.fit_transform(
    labels_train.reshape(-1, 1)
)

y_test = encoder.transform(
    labels_test.reshape(-1, 1)
)


scaler = StandardScaler()

N_train, T, F = X_train.shape
N_test = X_test.shape[0]

X_train_2d = X_train.reshape(-1, F)
X_test_2d = X_test.reshape(-1, F)

scaler.fit(X_train_2d)

X_train_scaled = scaler.transform(X_train_2d).reshape(
    N_train, T, F
)

X_test_scaled = scaler.transform(X_test_2d).reshape(
    N_test, T, F
)


num_classes = y_train.shape[1]

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(T, F)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)


history = model.fit(
    X_train_scaled,
    y_train,
    epochs=30,
    batch_size=16,
    verbose=1,
)


train_loss, train_accuracy = model.evaluate(
    X_train_scaled,
    y_train,
    verbose=0
)

test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(
    f"Train Accuracy: {train_accuracy * 100:.2f}%\n"
    f"Test Accuracy: {test_accuracy * 100:.2f}%"
)


# Save pipeline

joblib.dump(
    encoder,
    gesture_model_dir / "encoder.pkl"
)

joblib.dump(
    scaler,
    gesture_model_dir / "scaler.pkl"
)

model.save(
    gesture_model_dir / "model.keras"
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1069 - loss: 2.0940  
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2013 - loss: 2.0389
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2642 - loss: 1.9988
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2830 - loss: 1.9577
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3270 - loss: 1.8436
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3836 - loss: 1.6876
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4717 - loss: 1.5067
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5723 - loss: 1.3418
Epoch 9/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6855 - loss: 1.1485
Epoch 10/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7233 - loss: 0.9912
Epoch 11/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7610 - loss: 0.8646
Epoch 12/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7862 - 